# Train Ultralytics YOLO from Scratch in Google Colab

This notebook trains a YOLO detector from randomly initialized architecture weights. It does not load pretrained `.pt` weights.

## Dataset format

```text
animals/
├── images/train
├── images/val
├── labels/train
├── labels/val
└── data.yaml
```

Each label file uses:

```text
class_id center_x center_y width height
```

All coordinates must be normalized from 0 to 1.

In [13]:
!pip install -q -U ultralytics pyyaml

In [14]:
import os
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA available: True
GPU: Tesla T4


In [21]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [22]:
DATA = '/content/drive/MyDrive/animals/animal_dataset.yaml'
RUNS_DIR = '/content/drive/MyDrive/yolo_runs'
RUN_NAME = 'animals_from_scratch'

assert os.path.exists(DATA), f'Dataset YAML not found: {DATA}'
print(DATA)

/content/drive/MyDrive/animals/animal_dataset.yaml


In [24]:
with open(DATA, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = Path(data.get('path', Path(DATA).parent))
if not dataset_root.is_absolute():
    dataset_root = Path(DATA).parent / dataset_root

train_dir = dataset_root / data['train']
val_dir = dataset_root / data['val']
names = data['names']

print('Dataset root:', dataset_root)
print('Training images:', len([p for p in train_dir.rglob('*') if p.is_file()]))
print('Validation images:', len([p for p in val_dir.rglob('*') if p.is_file()]))
print('Classes:', names)

Dataset root: /content/drive/MyDrive/animals
Training images: 1052
Validation images: 225
Classes: {0: 'buffalo', 1: 'elephant', 2: 'lion', 3: 'zebra'}


In [25]:
model = YOLO('yolo26n.yaml')
print('Model initialized from architecture YAML.')
print('No pretrained .pt weights were loaded.')

Model initialized from architecture YAML.
No pretrained .pt weights were loaded.


In [ ]:
device = 0 if torch.cuda.is_available() else 'cpu'

results = model.train(
    data=DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    workers=2,
    pretrained=False,
    patience=25,
    project=RUNS_DIR,
    name=RUN_NAME,
    plots=True,
    save=True,
    exist_ok=True
)

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/animals/animal_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=animals_f

In [ ]:
best_path = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_path)

metrics = best_model.val(
    data=DATA,
    imgsz=640,
    device=device,
    plots=True
)

print('Best weights:', best_path)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
TEST_IMAGE = '/content/drive/MyDrive/animals/test.jpg'

prediction = best_model.predict(
    source=TEST_IMAGE,
    conf=0.35,
    imgsz=640,
    device=device,
    save=True
)

prediction[0].show()

In [ ]:
for result in prediction:
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print({
            'class': result.names[class_id],
            'confidence': round(confidence, 4),
            'box': [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]
        })

## Optional: upload an image for inference

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_image = next(iter(uploaded))

uploaded_results = best_model.predict(
    source=uploaded_image,
    conf=0.35,
    imgsz=640,
    device=device,
    save=True
)

uploaded_results[0].show()

In [ ]:
EXPORT_DIR = f'{RUNS_DIR}/{RUN_NAME}/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

best_model.export(
    format='onnx',
    imgsz=640,
    project=EXPORT_DIR,
    name='animals_detector_onnx'
)